# Fase 1: Data Engineering

Abran el archivo .json usando la librería json de Python. Extraigan los datos para construir una única tabla maestra (DataFrame) con 3 columnas: Origen, Destino y Relacion.

In [ ]:
import json
import pandas as pd

with open('api_troll_farm.json') as f:
    data = json.load(f)

df = pd.json_normalize(data)
df['account_id'] = df['account_id'].str.upper()

**Reto A (La Capa Social):** Extraigan los "retweets". Si Usuario_A retuitea a Usuario_B, eso es una relación tipo "Retweet". Ignoren auto-retweets. (Ojo con las minúsculas/mayúsculas).

In [ ]:
rt = df[['account_id', 'activity.retweets']].explode('activity.retweets').dropna()
rt.columns = ['Origen', 'Destino']
rt['Destino'] = rt['Destino'].str.upper()
rt = rt[rt['Origen'] != rt['Destino']].assign(Relacion='Retweet')

**Reto B (La Capa de Infraestructura):** En el JSON, la IP y la Tarjeta son metadatos individuales. Deberán ingeniárselas (recomendación: extraigan los metadatos a un DataFrame y hagan un merge del dataframe consigo mismo) para conectar a los usuarios. Si Usuario_A y Usuario_B tienen la misma Tarjeta, agreguen una fila a su tabla con relación tipo "Misma_Tarjeta".

In [ ]:
meta = df[['account_id', 'metadata.payment_hash']].dropna().drop_duplicates()
tj = meta.merge(meta, on='metadata.payment_hash')
tj = tj[tj['account_id_x'] != tj['account_id_y']]
tj = pd.DataFrame({'Origen': tj['account_id_x'], 'Destino': tj['account_id_y'], 'Relacion': 'Misma_Tarjeta'})

tabla = pd.concat([rt, tj], ignore_index=True).drop_duplicates()
tabla

# Fase 2: El Grafo Multicapa

Pasen su tabla maestra purificada a NetworkX utilizando nx.MultiDiGraph(). Comprueben que su grafo cargó correctamente imprimiendo el número total de Nodos y Aristas.

In [ ]:
import networkx as nx

MG = nx.from_pandas_edgelist(tabla, 'Origen', 'Destino', edge_attr='Relacion', create_using=nx.MultiDiGraph)
print(MG.number_of_nodes(), MG.number_of_edges())

# Fase 3: El Slicing

Filtren su MultiGrafo para extraer exclusivamente las aristas cuya Relación sea "Misma_Tarjeta". Metan esas aristas en un grafo no dirigido simple (nx.Graph()). Usen la función nx.connected_components() sobre este nuevo sub-grafo para encontrar a la tribu coludida financieramente.

In [ ]:
G_tj = nx.Graph([(u, v) for u, v, d in MG.edges(data=True) if d['Relacion'] == 'Misma_Tarjeta'])
anillo = list(nx.connected_components(G_tj))
print(anillo)
print(meta[meta['account_id'].isin(anillo[0])]['metadata.payment_hash'].unique())

# El Veredicto

**¿Cuáles son los nombres exactos (account_id) de los bots que conforman el anillo de fraude?**

El anillo lo forman TROLL_BOT_1, TROLL_BOT_2, TROLL_BOT_3, TROLL_BOT_4, TROLL_BOT_5 y TROLL_BOT_6.

**¿Cuál es el Hash de la tarjeta de crédito comprometida que el equipo de soporte técnico debe bloquear ahora mismo?**

Se debe bloquear la tarjeta VISA-4455-XXXX-9090.

**Si tuvieran que entrenar un modelo de predicción, ¿cómo le pasarían este descubrimiento al modelo tabular (La matriz X de entrenamiento)?**

Añadiría a cada cuenta una variable con cuántas cuentas comparten su tarjeta, o un indicador binario de tarjeta compartida, para que el modelo reciba la relación del grafo como una columna numérica.